# IMPORT LIBRARY

NLTK dipakai untuk memahami, membersihkan, dan mengolah teks sebelum masuk ke model machine learning.

nltk itu memecah kalimat menjadi kata

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [2]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dimas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dimas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dimas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
df = pd.read_csv('aug_train.csv')
df.head()

,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target
0,8949,city_103,0.920,Male,Has relevent experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevent experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevent experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0
3,33241,city_115,0.789,NaN,No relevent experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1.0
4,666,city_162,0.767,Male,Has relevent experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0.0


# Mengetahui kolom apa aja

In [4]:
df.columns

Index(['enrollee_id', 'city', 'city_development_index', 'gender',
       'relevent_experience', 'enrolled_university', 'education_level',
       'major_discipline', 'experience', 'company_size', 'company_type',
       'last_new_job', 'training_hours', 'target'],
      dtype='object')

# Menggabungkan Beberapa kolom

In [5]:
df['text'] = df.astype(str).agg(' '.join, axis=1)

# Text Preprocessing

In [6]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess)

# Define Keywords (Rule-Based)

In [7]:
keywords = {
    "Data Scientist": ["python", "machine learning", "data analysis", "pandas", "numpy"],
    "Web Developer": ["html", "css", "javascript", "react", "php"],
    "AI Engineer": ["deep learning", "tensorflow", "keras", "ai", "nlp"],
}

# Keyword Matching Function

In [8]:
def classify_role(text):
    for role, keys in keywords.items():
        for key in keys:
            if key in text:
                return role
    return "Other"

df['role_rule_based'] = df['clean_text'].apply(classify_role)

# Machine Learning Approach (TF-IDF + Random Forest)

Feature Extraction

In [9]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['clean_text'])

Label (misal pakai target)

In [10]:
y = df['target']

# Split Data

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train Model

In [12]:
model = RandomForestClassifier()
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# Evaluasi

In [13]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.79      0.94      0.86      2900
         1.0       0.53      0.20      0.29       932

    accuracy                           0.76      3832
   macro avg       0.66      0.57      0.57      3832
weighted avg       0.72      0.76      0.72      3832



# Testing Resume Baru

In [14]:
sample_resume = """
I have experience in Python, machine learning, data analysis, and SQL.
"""

clean = preprocess(sample_resume)
vector = tfidf.transform([clean])

prediction = model.predict(vector)

print("Predicted:", prediction)

Predicted: [0.]
